# **Sales Forecasting Dataset**

### **Load daily sales**

In [54]:
import pandas as pd
import numpy as np

daily_sales = pd.read_csv(
    "../data/processed/daily_sales.csv",
    parse_dates=["Date"]
)

daily_sales.head()

,Date,Revenue,Quantity,Orders,Customers
0,2009-12-01,54351.23,26098,119,91
1,2009-12-02,63172.58,31804,115,94
2,2009-12-03,73972.45,49221,124,106
3,2009-12-04,40582.32,21210,89,76
4,2009-12-05,9803.05,5119,30,26


### **Sort by date**

In [55]:
daily_sales = daily_sales.sort_values(
    "Date"
).reset_index(drop=True)

### **Create time features**

In [56]:
daily_sales["Year"] = daily_sales["Date"].dt.year

daily_sales["Month"] = daily_sales["Date"].dt.month

daily_sales["DayOfWeek"] = daily_sales["Date"].dt.dayofweek

daily_sales["IsWeekend"] = (
    daily_sales["DayOfWeek"] >= 5
).astype(int)

### **Create Lag features**

**Lag 1** 

the day before revenue

In [57]:
daily_sales["Lag_1"] = (
    daily_sales["Revenue"].shift(1)
)

**Lag 7**

Revenue from 7 days ago

In [58]:
daily_sales["Lag_7"] = (
    daily_sales["Revenue"].shift(7)
)

**Lag 14**

In [59]:
daily_sales["Lag_14"] = (
    daily_sales["Revenue"].shift(14)
)

**Lag 28**

In [60]:
daily_sales["Lag_28"] = (
    daily_sales["Revenue"].shift(28)
)

### **Create rolling averages**

A 7 day rolling average tells the model what was the average revenue been recently

In [61]:
daily_sales["RollingMean_7"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(7)
    .mean()
)

daily_sales["RollingMean_14"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(14)
    .mean()
)

daily_sales["RollingMean_28"] = (
    daily_sales["Revenue"]
    .shift(1)
    .rolling(28)
    .mean()
)

### **Selecting the Forecasting columns**

In [62]:
forecast_features = [
    "Year",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "RollingMean_7",
    "RollingMean_14",
    "RollingMean_28"
]

forecast_target = "Revenue"

forecast_df = daily_sales[
    ["Date"] +
    forecast_features + 
    [forecast_target]
].copy()

### **Remove rows created by lagging**

**the first 28 days doesnt have enough information so it cannot exist for the first 28 days**

In [63]:
forecast_df = forecast_df.dropna().reset_index(
    drop=True
)

forecast_df.head()

,Date,Year,Month,DayOfWeek,IsWeekend,Lag_1,Lag_7,Lag_14,Lag_28,RollingMean_7,RollingMean_14,RollingMean_28,Revenue
0,2010-01-12,2010,1,1,0,37886.41,13445.80,52545.55,54351.23,29444.998571,28452.942857,36735.676429,40786.98
1,2010-01-13,2010,1,2,0,40786.98,19114.34,30638.25,63172.58,33350.881429,27613.045000,36251.238929,22470.18
2,2010-01-14,2010,1,3,0,22470.18,8736.75,42470.29,73972.45,33830.287143,27029.611429,34797.581786,31550.90
3,2010-01-15,2010,1,4,0,31550.90,73039.89,11382.54,40582.32,37089.451429,26249.655000,33282.526429,13507.90
4,2010-01-17,2010,1,6,1,13507.90,30293.95,16322.95,9803.05,28584.881429,26401.466429,32315.582857,17393.20


# **Customer Segmentation Dataset**

In [64]:
sales_df = pd.read_csv(
    "../data/processed/valid_sales.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Invoice": "string"}
)

sales_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,Revenue,Year,Month,Day,DayOfWeek,Hour,IsWeekend
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4,2009,12,1,1,7,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,2009,12,1,1,7,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,2009,12,1,1,7,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8,2009,12,1,1,7,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0,2009,12,1,1,7,False


### **Calculate reference date**

In [65]:
reference_date = (
    sales_df["InvoiceDate"].max()
    + pd.Timedelta(days=1)
)

### **Create RFM**

**RFM analysis is a data-driven marketing method used to evaluate and group customers based on their past purchasing behavior**

**R - Recency**
       
        How many days since the customers last purchase ?
 
**F - Frequency**
       
        How many orders did the customer make in a specific period ?
 
**M - Monetary**
      
        How much money did the customer spend ?

drop missing customer id

In [66]:
rfm_sales = sales_df.dropna(
    subset=["Customer ID"]
).copy()

In [67]:
reference_date = rfm_sales["InvoiceDate"].max() + pd.Timedelta(days=1)

print("Reference Date:", reference_date)

Reference Date: 2011-12-10 12:50:00


In [68]:
rfm_df = (
    sales_df
    .groupby("Customer ID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x: (
                reference_date - x.max()
            ).days
        ),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum")
    )
    .reset_index()
)

rfm_df.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [69]:
rfm_df.shape

(5878, 4)

### **Add average order value**

In [70]:
rfm_df["AverageOrderValue"] = (
    rfm_df["Monetary"] /
    rfm_df["Frequency"]
)

### **Add numbers of unique products**

In [71]:
unique_products = (
    sales_df
    .groupby("Customer ID")["StockCode"]
    .nunique()
    .reset_index(
        name="UniqueProducts"
    )
)

rfm_df = rfm_df.merge(
    unique_products,
    on="Customer ID",
    how="left"
)

rfm_df.head()

,Customer ID,Recency,Frequency,Monetary,AverageOrderValue,UniqueProducts
0,12346.0,326,12,77556.46,6463.038333,27
1,12347.0,2,8,4921.53,615.191250,126
2,12348.0,75,5,2019.40,403.880000,25
3,12349.0,19,4,4428.69,1107.172500,138
4,12350.0,310,1,334.40,334.400000,17


# **Purchase Prediction Dataset**

**to help predict that a customer would buy something during the following 30 days**

In [72]:
cutoff_date = (
    sales_df["InvoiceDate"].max()
    - pd.Timedelta(days=30)
)

historical_df = sales_df[
    sales_df["InvoiceDate"] <= cutoff_date
].copy()

historical_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,Revenue,Year,Month,Day,DayOfWeek,Hour,IsWeekend
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4,2009,12,1,1,7,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,2009,12,1,1,7,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0,2009,12,1,1,7,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8,2009,12,1,1,7,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0,2009,12,1,1,7,False


future 30 day data 

In [73]:
future_df = sales_df[
    (sales_df["InvoiceDate"] > cutoff_date) &
    (
        sales_df["InvoiceDate"]
        <= cutoff_date + pd.Timedelta(days=30)
    )
].copy()

future_df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,Revenue,Year,Month,Day,DayOfWeek,Hour,IsWeekend
920673,575317,23544,WALL ART MID CENTURY MODERN,2,2011-11-09 13:12:00,8.25,17402.0,United Kingdom,False,16.50,2011,11,9,2,13,False
920674,575317,21790,VINTAGE SNAP CARDS,12,2011-11-09 13:12:00,0.85,17402.0,United Kingdom,False,10.20,2011,11,9,2,13,False
920675,575317,21791,VINTAGE HEADS AND TAILS CARD GAME,12,2011-11-09 13:12:00,1.25,17402.0,United Kingdom,False,15.00,2011,11,9,2,13,False
920676,575317,22619,SET OF 6 SOLDIER SKITTLES,80,2011-11-09 13:12:00,3.39,17402.0,United Kingdom,False,271.20,2011,11,9,2,13,False
920677,575318,23318,BOX OF 6 MINI VINTAGE CRACKERS,12,2011-11-09 13:17:00,2.49,14921.0,United Kingdom,False,29.88,2011,11,9,2,13,False


In [74]:
reference_date = cutoff_date + pd.Timedelta(days=1)

purchase_features = (
    historical_df
    .groupby("Customer ID")
    .agg(
        Recency=(
            "InvoiceDate",
            lambda x: (
                reference_date - x.max()
            ).days
        ),
        Frequency=("Invoice", "nunique"),
        Monetary=("Revenue", "sum"),
        UniqueProducts=("StockCode", "nunique")
    )
    .reset_index()
)

purchase_features.head()

,Customer ID,Recency,Frequency,Monetary,UniqueProducts
0,12346.0,296,12,77556.46,27
1,12347.0,10,7,4696.71,123
2,12348.0,45,5,2019.40,25
3,12349.0,378,3,2671.14,90
4,12350.0,280,1,334.40,17


In [75]:
purchase_features["AverageOrderValue"] = (
    purchase_features["Monetary"] /
    purchase_features["Frequency"]
)

purchase_features.head()

,Customer ID,Recency,Frequency,Monetary,UniqueProducts,AverageOrderValue
0,12346.0,296,12,77556.46,27,6463.038333
1,12347.0,10,7,4696.71,123,670.958571
2,12348.0,45,5,2019.40,25,403.880000
3,12349.0,378,3,2671.14,90,890.380000
4,12350.0,280,1,334.40,17,334.400000


### **Create the target**

In [76]:
future_customers = (
    future_df["Customer ID"]
    .dropna()
    .unique()
)

print("Unique future customers:", len(future_customers))
print(future_customers[:5])

Unique future customers: 1648
[17402. 14921. 14836. 17011. 15311.]


In [77]:
purchase_features["PurchasedNext30Days"] = (
    purchase_features["Customer ID"]
    .isin(future_customers)
    .astype(int)
)

purchase_features.head()

,Customer ID,Recency,Frequency,Monetary,UniqueProducts,AverageOrderValue,PurchasedNext30Days
0,12346.0,296,12,77556.46,27,6463.038333,0
1,12347.0,10,7,4696.71,123,670.958571,1
2,12348.0,45,5,2019.40,25,403.880000,0
3,12349.0,378,3,2671.14,90,890.380000,1
4,12350.0,280,1,334.40,17,334.400000,0


### **Chronological split**

In [78]:
forecast_df = forecast_df.sort_values("Date")

n = len(forecast_df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

train_df = forecast_df.iloc[:train_end]
validation_df = forecast_df.iloc[train_end:validation_end]
test_df = forecast_df.iloc[validation_end:]

In [79]:
FEATURES = [
    "Year",
    "Month",
    "DayOfWeek",
    "IsWeekend",
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "RollingMean_7",
    "RollingMean_14",
    "RollingMean_28"
]

TARGET = "Revenue"